In [ ]:
!pip install ydata-profiling

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.4/400.4 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 3.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import unicodedata
import re
from collections import defaultdict

In [ ]:
FILE = "restaurant_ratings.csv"

In [ ]:
df = pd.read_csv(FILE, on_bad_lines='skip', quoting=3, encoding='utf-8', engine='python')
issues = defaultdict(list)

print("=" * 60)
print(f"  CSV QUALITY REPORT — {FILE}")
print(f"  {df.shape[0]:,} rows × {df.shape[1]} columns")
print("=" * 60)

  CSV QUALITY REPORT — restaurant_ratings.csv
  2,557 rows × 5 columns


In [ ]:
print("\n[1] MISSING VALUES")
nulls = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(2)
has_nulls = False
for col in df.columns:
    if nulls[col] > 0:
        print(f"    ✗ '{col}': {nulls[col]:,} nulls ({null_pct[col]}%)")
        issues[col].append(f"{nulls[col]} missing values")
        has_nulls = True
if not has_nulls:
    print("    ✓ No missing values")


[1] MISSING VALUES
    ✗ 'restaurant_name': 2,429 nulls (94.99%)
    ✗ 'location': 2,452 nulls (95.89%)
    ✗ 'rating': 2,475 nulls (96.79%)
    ✗ 'votes': 2,510 nulls (98.16%)
    ✗ 'rating_text': 2,534 nulls (99.1%)


In [ ]:
print("\n[2] DUPLICATE ROWS")
dupe_count = df.duplicated().sum()
if dupe_count:
    print(f"    ✗ {dupe_count:,} fully duplicate rows found")
    issues["_dataset"].append(f"{dupe_count} duplicate rows")
else:
    print("    ✓ No duplicate rows")


[2] DUPLICATE ROWS
    ✗ 2,487 fully duplicate rows found


In [ ]:
print("\n[3] LEADING / TRAILING WHITESPACE")
str_cols = df.select_dtypes(include="object").columns
for col in str_cols:
    ws = df[col].str.strip().ne(df[col]).sum()
    if ws:
        print(f"    ✗ '{col}': {ws:,} cells have extra whitespace")
        issues[col].append(f"{ws} whitespace issues")
    else:
        print(f"    ✓ '{col}': clean")


[3] LEADING / TRAILING WHITESPACE
    ✗ 'restaurant_name': 125 cells have extra whitespace
    ✗ 'location': 104 cells have extra whitespace
    ✗ 'rating': 81 cells have extra whitespace
    ✗ 'votes': 47 cells have extra whitespace
    ✗ 'rating_text': 23 cells have extra whitespace


In [ ]:
print("\n[4] INCONSISTENT CASING")
for col in str_cols:
    unique_raw = df[col].str.strip().dropna().unique()
    groups = defaultdict(list)
    for v in unique_raw:
        groups[v.lower()].append(v)
    mixed = {k: v for k, v in groups.items() if len(v) > 1}
    if mixed:
        examples = list(mixed.values())[:3]
        ex_str = " | ".join([str(e) for e in examples])
        print(f"    ✗ '{col}': {len(mixed):,} values with inconsistent casing  e.g. {ex_str}")
        issues[col].append(f"{len(mixed)} casing inconsistencies")
    else:
        print(f"    ✓ '{col}': consistent")


[4] INCONSISTENT CASING
    ✓ 'restaurant_name': consistent
    ✓ 'location': consistent
    ✓ 'rating': consistent
    ✓ 'votes': consistent
    ✓ 'rating_text': consistent


In [ ]:
print("\n[5] ENCODING / GARBLED TEXT")
def has_encoding_issue(text):
    if not isinstance(text, str):
        return False
    # Flag classic mojibake multi-byte garbage sequences
    return bool(re.search(r'[ãâïÃÂ]{2,}|[\x80-\x9f]|â€|Â·|Ã©|Ã¨|Ã¼', text))

for col in str_cols:
    bad = df[col].apply(has_encoding_issue).sum()
    if bad:
        example = df[col][df[col].apply(has_encoding_issue)].iloc[0]
        print(f"    ✗ '{col}': {bad:,} cells with encoding issues  e.g. '{example}'")
        issues[col].append(f"{bad} encoding issues")
    else:
        print(f"    ✓ '{col}': clean")


[5] ENCODING / GARBLED TEXT
    ✗ 'restaurant_name': 1 cells with encoding issues  e.g. ' I do have strong opinion when it comes to Neer Dosa ?)\n\nIÃ\x83Ã\x83Ã\x82Ã\x82Ã\x83Ã\x82Ã\x82Ã\x92d say this is one of the best breakfast places in the area. A must visit.')]"'
    ✓ 'location': clean
    ✓ 'rating': clean
    ✓ 'votes': clean
    ✗ 'rating_text': 3 cells with encoding issues  e.g. ' 'RATED\n  Chinese is everyoneÃ\x83Ã\x83Ã\x82Ã\x82Ã\x83Ã\x82Ã\x82Ã\x92s favourite??\n\nYummy Hakka noodles and chilli paneer? and for the dessert yummy Dassan with ice (which I loved it a lot)???\n\nDonÃ\x83Ã\x83Ã\x82Ã\x82Ã\x83Ã\x82Ã\x82Ã\x92t forget to follow my Instagram page:chalo_khaayein')]"'


In [ ]:
print("\n[6] NUMERIC OUTLIERS (IQR method)")
num_cols = df.select_dtypes(include="number").columns
for col in num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    if len(outliers):
        print(f"    ✗ '{col}': {len(outliers):,} outliers  (expected {lower:.1f}–{upper:.1f}, got min={df[col].min()}, max={df[col].max()})")
        issues[col].append(f"{len(outliers)} outliers")
    else:
        print(f"    ✓ '{col}': no outliers")


[6] NUMERIC OUTLIERS (IQR method)


In [ ]:
print("\n[7] MIXED DATA TYPES IN STRING COLUMNS")
for col in str_cols:
    numeric_looking = df[col].dropna().str.match(r'^\s*-?\d+\.?\d*\s*$').sum()
    total = df[col].notna().sum()
    if 0 < numeric_looking < total:
        print(f"    ✗ '{col}': {numeric_looking:,} cells look numeric inside a text column")
        issues[col].append(f"{numeric_looking} cells appear numeric")
    else:
        print(f"    ✓ '{col}': consistent type")

# ── 8. NEAR-DUPLICATE VALUES (within string cols) ───────────────────────────
print("\n[8] NEAR-DUPLICATES (values differing only by spaces/case)")
for col in str_cols:
    normalized = df[col].str.strip().str.lower()
    dupe_mask = normalized.duplicated(keep=False)
    raw_dupes = df[col][dupe_mask & df[col].str.strip().str.lower().ne(df[col])]
    count = raw_dupes.nunique()
    if count:
        print(f"    ✗ '{col}': {count:,} near-duplicate values (same after normalizing)")
        issues[col].append(f"{count} near-duplicates")
    else:
        print(f"    ✓ '{col}': no near-duplicates detected")

# ── SUMMARY ─────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  SUMMARY OF ISSUES")
print("=" * 60)
if not issues:
    print("  ✓ Dataset looks clean!")
else:
    all_issues = 0
    for key, vals in issues.items():
        label = "Dataset" if key == "_dataset" else f"Column '{key}'"
        for v in vals:
            print(f"  • {label}: {v}")
            all_issues += 1
    print(f"\n  Total issue types found: {all_issues}")
print("=" * 60)


[7] MIXED DATA TYPES IN STRING COLUMNS
    ✓ 'restaurant_name': consistent type
    ✓ 'location': consistent type
    ✓ 'rating': consistent type
    ✓ 'votes': consistent type
    ✓ 'rating_text': consistent type

[8] NEAR-DUPLICATES (values differing only by spaces/case)
    ✗ 'restaurant_name': 29 near-duplicate values (same after normalizing)
    ✗ 'location': 24 near-duplicate values (same after normalizing)
    ✗ 'rating': 21 near-duplicate values (same after normalizing)
    ✗ 'votes': 11 near-duplicate values (same after normalizing)
    ✗ 'rating_text': 6 near-duplicate values (same after normalizing)

  SUMMARY OF ISSUES
  • Column 'restaurant_name': 2429 missing values
  • Column 'restaurant_name': 125 whitespace issues
  • Column 'restaurant_name': 1 encoding issues
  • Column 'restaurant_name': 29 near-duplicates
  • Column 'location': 2452 missing values
  • Column 'location': 104 whitespace issues
  • Column 'location': 24 near-duplicates
  • Column 'rating': 2475 missi

In [ ]:
df = pd.read_csv(FILE, on_bad_lines='skip', quoting=3, engine='python')
print(df.head(10))
print(df.shape)

In [ ]:
with open(FILE, 'r', encoding='utf-8', errors='replace') as f:
    for i, line in enumerate(f):
        print(repr(line))
        if i > 10:
            break

'restaurant_name,location,rating,votes,rating_text\n'
' jalsa,  Banashankari ,4.1/5,775.0,"[(\'Rated 4.0\', \'RATED\\n  A beautiful place to dine in.The interiors take you back to the Mughal era. The lightings are just perfect.We went there on the occasion of Christmas and so they had only limited items available. But the taste and service was not compromised at all.The only complaint is that the breads could have been better.Would surely like to come here again.\'), (\'Rated 4.0\', \'RATED\\n  I was here for dinner with my family on a weekday. The restaurant was completely empty. Ambience is good with some good old hindi music. Seating arrangement are good too. We ordered masala papad, panner and baby corn starters, lemon and corrionder soup, butter roti, olive and chilli paratha. Food was fresh and good, service is good too. Good for family hangout.\\nCheers\'), (\'Rated 2.0\', \'RATED\\n  Its a restaurant near to Banashankari BDA. Me along with few of my office friends visited to ha

In [ ]:
import pandas as pd
import re

rows = []

with open('restaurant_ratings.csv', 'r', encoding='utf-8', errors='replace') as f:
    next(f)  # skip header
    content = f.read()

# Split on lines that start a new restaurant record
# Each real row starts with a restaurant name before the first comma
pattern = re.compile(
    r'^(.*?),(.*?),([\d.]+/5),([\d.]+|NaN),("(?:.|\n)*?")\s*$',
    re.MULTILINE
)

for match in pattern.finditer(content):
    rows.append({
        'restaurant_name': match.group(1).strip(),
        'location':        match.group(2).strip(),
        'rating':          match.group(3).strip(),
        'votes':           match.group(4).strip(),
        'rating_text':     match.group(5).strip()
    })

df = pd.DataFrame(rows)

# Clean up
df['votes'] = pd.to_numeric(df['votes'], errors='coerce')
df['rating'] = df['rating'].str.replace('/5', '').astype(float)
df['restaurant_name'] = df['restaurant_name'].str.strip().str.title()
df['location'] = df['location'].str.strip().str.title()
df = df.drop_duplicates()

print(df.shape)
print(df.head())
df.to_csv('restaurant_ratings_fixed.csv', index=False)

(3068, 5)
         restaurant_name      location  rating  votes  \
0                  Jalsa  Banashankari     4.1  775.0   
1         Spice Elephant  Banashankari     4.1  787.0   
2        San Churro Cafe  Banashankari     3.8  918.0   
3  Addhuri Udupi Bhojana  Banashankari     3.7   88.0   
4          Grand Village  Basavanagudi     3.8  166.0   

                                         rating_text  
0  "[('Rated 4.0', 'RATED\n  A beautiful place to...  
1  "[('Rated 4.0', 'RATED\n  Had been here for di...  
2  "[('Rated 3.0', ""RATED\n  Ambience is not tha...  
3  "[('Rated 4.0', ""RATED\n  Great food and prop...  
4  "[('Rated 4.0', 'RATED\n  Very good restaurant...  


In [ ]:
import ast

def parse_reviews(text):
    try:
        # Remove the outer quotes and parse as Python literal
        cleaned = text.strip('"')
        reviews = ast.literal_eval(cleaned)
        # Return just the review texts, stripping "RATED\n  " prefix
        return [r[1].replace('RATED\n  ', '').strip() for r in reviews if isinstance(r, tuple)]
    except:
        return []

df['reviews'] = df['rating_text'].apply(parse_reviews)
df['review_count'] = df['reviews'].apply(len)

# Optional: explode into one row per review
df_exploded = df[['restaurant_name', 'location', 'rating', 'votes', 'reviews']]\
    .explode('reviews')\
    .rename(columns={'reviews': 'review'})\
    .reset_index(drop=True)

print(df_exploded.head())
print(f"\nTotal individual reviews: {len(df_exploded):,}")

  restaurant_name      location  rating  votes  \
0           Jalsa  Banashankari     4.1  775.0   
1           Jalsa  Banashankari     4.1  775.0   
2           Jalsa  Banashankari     4.1  775.0   
3           Jalsa  Banashankari     4.1  775.0   
4           Jalsa  Banashankari     4.1  775.0   

                                              review  
0  A beautiful place to dine in.The interiors tak...  
1  I was here for dinner with my family on a week...  
2  Its a restaurant near to Banashankari BDA. Me ...  
3  We went here on a weekend and one of us had th...  
4  The best thing about the place is itÃÃÃÃ...  

Total individual reviews: 4,355


In [ ]:
import pandas as pd
import unicodedata
import re
from collections import defaultdict

# ── CONFIG ──────────────────────────────────────────────────────────────────
FILE = "restaurant_ratings_fixed.csv"  # ← change to your filename
# ────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(FILE)
issues = defaultdict(list)

print("=" * 60)
print(f"  CSV QUALITY REPORT — {FILE}")
print(f"  {df.shape[0]:,} rows × {df.shape[1]} columns")
print("=" * 60)

# ── 1. MISSING VALUES ────────────────────────────────────────────────────────
print("\n[1] MISSING VALUES")
nulls = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(2)
has_nulls = False
for col in df.columns:
    if nulls[col] > 0:
        print(f"    ✗ '{col}': {nulls[col]:,} nulls ({null_pct[col]}%)")
        issues[col].append(f"{nulls[col]} missing values")
        has_nulls = True
if not has_nulls:
    print("    ✓ No missing values")

# ── 2. DUPLICATE ROWS ────────────────────────────────────────────────────────
print("\n[2] DUPLICATE ROWS")
dupe_count = df.duplicated().sum()
if dupe_count:
    print(f"    ✗ {dupe_count:,} fully duplicate rows found")
    issues["_dataset"].append(f"{dupe_count} duplicate rows")
else:
    print("    ✓ No duplicate rows")

# ── 3. WHITESPACE ISSUES (string columns) ───────────────────────────────────
print("\n[3] LEADING / TRAILING WHITESPACE")
str_cols = df.select_dtypes(include="object").columns
for col in str_cols:
    ws = df[col].str.strip().ne(df[col]).sum()
    if ws:
        print(f"    ✗ '{col}': {ws:,} cells have extra whitespace")
        issues[col].append(f"{ws} whitespace issues")
    else:
        print(f"    ✓ '{col}': clean")

# ── 4. INCONSISTENT CASING ──────────────────────────────────────────────────
print("\n[4] INCONSISTENT CASING")
for col in str_cols:
    unique_raw = df[col].str.strip().dropna().unique()
    groups = defaultdict(list)
    for v in unique_raw:
        groups[v.lower()].append(v)
    mixed = {k: v for k, v in groups.items() if len(v) > 1}
    if mixed:
        examples = list(mixed.values())[:3]
        ex_str = " | ".join([str(e) for e in examples])
        print(f"    ✗ '{col}': {len(mixed):,} values with inconsistent casing  e.g. {ex_str}")
        issues[col].append(f"{len(mixed)} casing inconsistencies")
    else:
        print(f"    ✓ '{col}': consistent")

# ── 5. ENCODING ISSUES ──────────────────────────────────────────────────────
print("\n[5] ENCODING / GARBLED TEXT")
def has_encoding_issue(text):
    if not isinstance(text, str):
        return False
    # Flag classic mojibake multi-byte garbage sequences
    return bool(re.search(r'[ãâïÃÂ]{2,}|[\x80-\x9f]|â€|Â·|Ã©|Ã¨|Ã¼', text))

for col in str_cols:
    bad = df[col].apply(has_encoding_issue).sum()
    if bad:
        example = df[col][df[col].apply(has_encoding_issue)].iloc[0]
        print(f"    ✗ '{col}': {bad:,} cells with encoding issues  e.g. '{example}'")
        issues[col].append(f"{bad} encoding issues")
    else:
        print(f"    ✓ '{col}': clean")

# ── 6. NUMERIC OUTLIERS ─────────────────────────────────────────────────────
print("\n[6] NUMERIC OUTLIERS (IQR method)")
num_cols = df.select_dtypes(include="number").columns
for col in num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    if len(outliers):
        print(f"    ✗ '{col}': {len(outliers):,} outliers  (expected {lower:.1f}–{upper:.1f}, got min={df[col].min()}, max={df[col].max()})")
        issues[col].append(f"{len(outliers)} outliers")
    else:
        print(f"    ✓ '{col}': no outliers")

# ── 7. MIXED DATA TYPES ─────────────────────────────────────────────────────
print("\n[7] MIXED DATA TYPES IN STRING COLUMNS")
for col in str_cols:
    numeric_looking = df[col].dropna().str.match(r'^\s*-?\d+\.?\d*\s*$').sum()
    total = df[col].notna().sum()
    if 0 < numeric_looking < total:
        print(f"    ✗ '{col}': {numeric_looking:,} cells look numeric inside a text column")
        issues[col].append(f"{numeric_looking} cells appear numeric")
    else:
        print(f"    ✓ '{col}': consistent type")

# ── 8. NEAR-DUPLICATE VALUES (within string cols) ───────────────────────────
print("\n[8] NEAR-DUPLICATES (values differing only by spaces/case)")
for col in str_cols:
    normalized = df[col].str.strip().str.lower()
    dupe_mask = normalized.duplicated(keep=False)
    raw_dupes = df[col][dupe_mask & df[col].str.strip().str.lower().ne(df[col])]
    count = raw_dupes.nunique()
    if count:
        print(f"    ✗ '{col}': {count:,} near-duplicate values (same after normalizing)")
        issues[col].append(f"{count} near-duplicates")
    else:
        print(f"    ✓ '{col}': no near-duplicates detected")

# ── SUMMARY ─────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  SUMMARY OF ISSUES")
print("=" * 60)
if not issues:
    print("  ✓ Dataset looks clean!")
else:
    all_issues = 0
    for key, vals in issues.items():
        label = "Dataset" if key == "_dataset" else f"Column '{key}'"
        for v in vals:
            print(f"  • {label}: {v}")
            all_issues += 1
    print(f"\n  Total issue types found: {all_issues}")
print("=" * 60)

  CSV QUALITY REPORT — restaurant_ratings_fixed.csv
  3,068 rows × 5 columns

[1] MISSING VALUES
    ✓ No missing values

[2] DUPLICATE ROWS
    ✓ No duplicate rows

[3] LEADING / TRAILING WHITESPACE
    ✓ 'restaurant_name': clean
    ✓ 'location': clean
    ✓ 'rating_text': clean

[4] INCONSISTENT CASING
    ✓ 'restaurant_name': consistent
    ✓ 'location': consistent
    ✓ 'rating_text': consistent

[5] ENCODING / GARBLED TEXT
    ✗ 'restaurant_name': 21 cells with encoding issues  e.g. 'CafãÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ© Down The Alley'
    ✓ 'location': clean
    ✗ 'rating_text': 1,003 cells with encoding issues  e.g. '"[('Rated 4.0', 'RATED\n  A beautiful place to dine in.The interiors take you back to the Mughal era. The lightings are just perfect.We went there on the occasion of Christmas and so they had only limited items available. But the taste and service was not compromised at all.The only complaint is that the breads could have been better.Would surely like to come here a

In [ ]:
import re, unicodedata

# 1. Fix encoding in restaurant_name (21 cells)
def fix_mojibake(text):
    if not isinstance(text, str):
        return text
    try:
        # Try to decode as latin-1 re-encoded as utf-8
        return text.encode('latin-1').decode('utf-8')
    except:
        return unicodedata.normalize('NFKC', text)

df['restaurant_name'] = df['restaurant_name'].apply(fix_mojibake)

# 2. Drop rows where restaurant_name is numeric (3 rows - likely parsing artifacts)
df = df[~df['restaurant_name'].str.match(r'^\s*[\d.]+\s*$')]

# 3. Fix encoding in rating_text (the Ã\x83 mojibake pattern)
df['rating_text'] = df['rating_text'].apply(fix_mojibake)

In [ ]:
import pandas as pd
import unicodedata
import re
from collections import defaultdict

# ── CONFIG ──────────────────────────────────────────────────────────────────
FILE = "restaurant_ratings_fixed.csv"  # ← change to your filename
# ────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(FILE)
issues = defaultdict(list)

print("=" * 60)
print(f"  CSV QUALITY REPORT — {FILE}")
print(f"  {df.shape[0]:,} rows × {df.shape[1]} columns")
print("=" * 60)

# ── 1. MISSING VALUES ────────────────────────────────────────────────────────
print("\n[1] MISSING VALUES")
nulls = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(2)
has_nulls = False
for col in df.columns:
    if nulls[col] > 0:
        print(f"    ✗ '{col}': {nulls[col]:,} nulls ({null_pct[col]}%)")
        issues[col].append(f"{nulls[col]} missing values")
        has_nulls = True
if not has_nulls:
    print("    ✓ No missing values")

# ── 2. DUPLICATE ROWS ────────────────────────────────────────────────────────
print("\n[2] DUPLICATE ROWS")
dupe_count = df.duplicated().sum()
if dupe_count:
    print(f"    ✗ {dupe_count:,} fully duplicate rows found")
    issues["_dataset"].append(f"{dupe_count} duplicate rows")
else:
    print("    ✓ No duplicate rows")

# ── 3. WHITESPACE ISSUES (string columns) ───────────────────────────────────
print("\n[3] LEADING / TRAILING WHITESPACE")
str_cols = df.select_dtypes(include="object").columns
for col in str_cols:
    ws = df[col].str.strip().ne(df[col]).sum()
    if ws:
        print(f"    ✗ '{col}': {ws:,} cells have extra whitespace")
        issues[col].append(f"{ws} whitespace issues")
    else:
        print(f"    ✓ '{col}': clean")

# ── 4. INCONSISTENT CASING ──────────────────────────────────────────────────
print("\n[4] INCONSISTENT CASING")
for col in str_cols:
    unique_raw = df[col].str.strip().dropna().unique()
    groups = defaultdict(list)
    for v in unique_raw:
        groups[v.lower()].append(v)
    mixed = {k: v for k, v in groups.items() if len(v) > 1}
    if mixed:
        examples = list(mixed.values())[:3]
        ex_str = " | ".join([str(e) for e in examples])
        print(f"    ✗ '{col}': {len(mixed):,} values with inconsistent casing  e.g. {ex_str}")
        issues[col].append(f"{len(mixed)} casing inconsistencies")
    else:
        print(f"    ✓ '{col}': consistent")

# ── 5. ENCODING ISSUES ──────────────────────────────────────────────────────
print("\n[5] ENCODING / GARBLED TEXT")
def has_encoding_issue(text):
    if not isinstance(text, str):
        return False
    # Flag classic mojibake multi-byte garbage sequences
    return bool(re.search(r'[ãâïÃÂ]{2,}|[\x80-\x9f]|â€|Â·|Ã©|Ã¨|Ã¼', text))

for col in str_cols:
    bad = df[col].apply(has_encoding_issue).sum()
    if bad:
        example = df[col][df[col].apply(has_encoding_issue)].iloc[0]
        print(f"    ✗ '{col}': {bad:,} cells with encoding issues  e.g. '{example}'")
        issues[col].append(f"{bad} encoding issues")
    else:
        print(f"    ✓ '{col}': clean")

# ── 6. NUMERIC OUTLIERS ─────────────────────────────────────────────────────
print("\n[6] NUMERIC OUTLIERS (IQR method)")
num_cols = df.select_dtypes(include="number").columns
for col in num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    if len(outliers):
        print(f"    ✗ '{col}': {len(outliers):,} outliers  (expected {lower:.1f}–{upper:.1f}, got min={df[col].min()}, max={df[col].max()})")
        issues[col].append(f"{len(outliers)} outliers")
    else:
        print(f"    ✓ '{col}': no outliers")

# ── 7. MIXED DATA TYPES ─────────────────────────────────────────────────────
print("\n[7] MIXED DATA TYPES IN STRING COLUMNS")
for col in str_cols:
    numeric_looking = df[col].dropna().str.match(r'^\s*-?\d+\.?\d*\s*$').sum()
    total = df[col].notna().sum()
    if 0 < numeric_looking < total:
        print(f"    ✗ '{col}': {numeric_looking:,} cells look numeric inside a text column")
        issues[col].append(f"{numeric_looking} cells appear numeric")
    else:
        print(f"    ✓ '{col}': consistent type")

# ── 8. NEAR-DUPLICATE VALUES (within string cols) ───────────────────────────
print("\n[8] NEAR-DUPLICATES (values differing only by spaces/case)")
for col in str_cols:
    normalized = df[col].str.strip().str.lower()
    dupe_mask = normalized.duplicated(keep=False)
    raw_dupes = df[col][dupe_mask & df[col].str.strip().str.lower().ne(df[col])]
    count = raw_dupes.nunique()
    if count:
        print(f"    ✗ '{col}': {count:,} near-duplicate values (same after normalizing)")
        issues[col].append(f"{count} near-duplicates")
    else:
        print(f"    ✓ '{col}': no near-duplicates detected")

# ── SUMMARY ─────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  SUMMARY OF ISSUES")
print("=" * 60)
if not issues:
    print("  ✓ Dataset looks clean!")
else:
    all_issues = 0
    for key, vals in issues.items():
        label = "Dataset" if key == "_dataset" else f"Column '{key}'"
        for v in vals:
            print(f"  • {label}: {v}")
            all_issues += 1
    print(f"\n  Total issue types found: {all_issues}")
print("=" * 60)

  CSV QUALITY REPORT — restaurant_ratings_fixed.csv
  3,068 rows × 5 columns

[1] MISSING VALUES
    ✓ No missing values

[2] DUPLICATE ROWS
    ✓ No duplicate rows

[3] LEADING / TRAILING WHITESPACE
    ✓ 'restaurant_name': clean
    ✓ 'location': clean
    ✓ 'rating_text': clean

[4] INCONSISTENT CASING
    ✓ 'restaurant_name': consistent
    ✓ 'location': consistent
    ✓ 'rating_text': consistent

[5] ENCODING / GARBLED TEXT
    ✗ 'restaurant_name': 21 cells with encoding issues  e.g. 'CafãÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ© Down The Alley'
    ✓ 'location': clean
    ✗ 'rating_text': 1,003 cells with encoding issues  e.g. '"[('Rated 4.0', 'RATED\n  A beautiful place to dine in.The interiors take you back to the Mughal era. The lightings are just perfect.We went there on the occasion of Christmas and so they had only limited items available. But the taste and service was not compromised at all.The only complaint is that the breads could have been better.Would surely like to come here a

In [ ]:
import re

def fix_mojibake(text):
    if not isinstance(text, str):
        return text
    # Keep decoding until it stops changing
    for _ in range(5):
        try:
            decoded = text.encode('latin-1').decode('utf-8')
            if decoded == text:
                break
            text = decoded
        except:
            break
    return text

df['restaurant_name'] = df['restaurant_name'].apply(fix_mojibake)
df['rating_text'] = df['rating_text'].apply(fix_mojibake)

In [ ]:
# Pick one bad name and inspect it
bad = df[df['restaurant_name'].str.contains('ã|Ã', case=False, na=False)].iloc[0]['restaurant_name']
print(repr(bad))

'Cafã\x83Â\x83Ã\x82Â\x83Ã\x83Â\x82Ã\x82Â\x83Ã\x83Â\x83Ã\x82Â\x82Ã\x83Â\x82Ã\x82Â© Down The Alley'


In [ ]:
def fix_triple_encoded(text):
    if not isinstance(text, str):
        return text
    try:
        return text.encode('latin-1').decode('utf-8')\
                   .encode('latin-1').decode('utf-8')\
                   .encode('latin-1').decode('utf-8')
    except:
        return text

df['restaurant_name'] = df['restaurant_name'].apply(fix_triple_encoded)
df['rating_text'] = df['rating_text'].apply(fix_triple_encoded)

# Verify
print(df[df['restaurant_name'].str.contains('Caf', na=False)]['restaurant_name'].head())

2                                       San Churro Cafe
8                                        Penthouse Cafe
10    CafãÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ© Down The A...
11                                         Cafe Shuffle
13                                           Caf-Eleven
Name: restaurant_name, dtype: object


In [ ]:
bad = df.loc[10, 'restaurant_name']
print(repr(bad))

# Then try brute-forcing more decode attempts
def fix_encoding_aggressive(text):
    if not isinstance(text, str):
        return text
    for attempts in range(6):
        try:
            decoded = text.encode('latin-1').decode('utf-8')
            if decoded == text:
                break
            text = decoded
        except:
            break
    return text

df['restaurant_name'] = df['restaurant_name'].apply(fix_encoding_aggressive)
print(df.loc[10, 'restaurant_name'])

'Cafã\x83Â\x83Ã\x82Â\x83Ã\x83Â\x82Ã\x82Â\x83Ã\x83Â\x83Ã\x82Â\x82Ã\x83Â\x82Ã\x82Â© Down The Alley'
CafãÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ© Down The Alley


In [ ]:
def fix_encoding_aggressive(text):
    if not isinstance(text, str):
        return text
    for _ in range(6):  # try up to 6 times
        try:
            decoded = text.encode('latin-1').decode('utf-8')
            if decoded == text:
                break
            text = decoded
        except:
            break
    return text

df['restaurant_name'] = df['restaurant_name'].apply(fix_encoding_aggressive)
df['rating_text'] = df['rating_text'].apply(fix_encoding_aggressive)

print(df.loc[10, 'restaurant_name'])

CafãÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ© Down The Alley


In [ ]:
df['restaurant_name'] = df['restaurant_name'].replace(
    df.loc[10, 'restaurant_name'],  # match exactly whatever garbled string is there
    'Café Down The Alley'
)

print(df.loc[10, 'restaurant_name'])  # should now show: Café Down The Alley

Café Down The Alley


In [ ]:
import pandas as pd
import unicodedata
import re
from collections import defaultdict

# ── CONFIG ──────────────────────────────────────────────────────────────────
FILE = "restaurant_ratings_fixed.csv"  # ← change to your filename
# ────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(FILE)
issues = defaultdict(list)

print("=" * 60)
print(f"  CSV QUALITY REPORT — {FILE}")
print(f"  {df.shape[0]:,} rows × {df.shape[1]} columns")
print("=" * 60)

# ── 1. MISSING VALUES ────────────────────────────────────────────────────────
print("\n[1] MISSING VALUES")
nulls = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(2)
has_nulls = False
for col in df.columns:
    if nulls[col] > 0:
        print(f"    ✗ '{col}': {nulls[col]:,} nulls ({null_pct[col]}%)")
        issues[col].append(f"{nulls[col]} missing values")
        has_nulls = True
if not has_nulls:
    print("    ✓ No missing values")

# ── 2. DUPLICATE ROWS ────────────────────────────────────────────────────────
print("\n[2] DUPLICATE ROWS")
dupe_count = df.duplicated().sum()
if dupe_count:
    print(f"    ✗ {dupe_count:,} fully duplicate rows found")
    issues["_dataset"].append(f"{dupe_count} duplicate rows")
else:
    print("    ✓ No duplicate rows")

# ── 3. WHITESPACE ISSUES (string columns) ───────────────────────────────────
print("\n[3] LEADING / TRAILING WHITESPACE")
str_cols = df.select_dtypes(include="object").columns
for col in str_cols:
    ws = df[col].str.strip().ne(df[col]).sum()
    if ws:
        print(f"    ✗ '{col}': {ws:,} cells have extra whitespace")
        issues[col].append(f"{ws} whitespace issues")
    else:
        print(f"    ✓ '{col}': clean")

# ── 4. INCONSISTENT CASING ──────────────────────────────────────────────────
print("\n[4] INCONSISTENT CASING")
for col in str_cols:
    unique_raw = df[col].str.strip().dropna().unique()
    groups = defaultdict(list)
    for v in unique_raw:
        groups[v.lower()].append(v)
    mixed = {k: v for k, v in groups.items() if len(v) > 1}
    if mixed:
        examples = list(mixed.values())[:3]
        ex_str = " | ".join([str(e) for e in examples])
        print(f"    ✗ '{col}': {len(mixed):,} values with inconsistent casing  e.g. {ex_str}")
        issues[col].append(f"{len(mixed)} casing inconsistencies")
    else:
        print(f"    ✓ '{col}': consistent")

# ── 5. ENCODING ISSUES ──────────────────────────────────────────────────────
print("\n[5] ENCODING / GARBLED TEXT")
def has_encoding_issue(text):
    if not isinstance(text, str):
        return False
    # Flag classic mojibake multi-byte garbage sequences
    return bool(re.search(r'[ãâïÃÂ]{2,}|[\x80-\x9f]|â€|Â·|Ã©|Ã¨|Ã¼', text))

for col in str_cols:
    bad = df[col].apply(has_encoding_issue).sum()
    if bad:
        example = df[col][df[col].apply(has_encoding_issue)].iloc[0]
        print(f"    ✗ '{col}': {bad:,} cells with encoding issues  e.g. '{example}'")
        issues[col].append(f"{bad} encoding issues")
    else:
        print(f"    ✓ '{col}': clean")

# ── 6. NUMERIC OUTLIERS ─────────────────────────────────────────────────────
print("\n[6] NUMERIC OUTLIERS (IQR method)")
num_cols = df.select_dtypes(include="number").columns
for col in num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    if len(outliers):
        print(f"    ✗ '{col}': {len(outliers):,} outliers  (expected {lower:.1f}–{upper:.1f}, got min={df[col].min()}, max={df[col].max()})")
        issues[col].append(f"{len(outliers)} outliers")
    else:
        print(f"    ✓ '{col}': no outliers")

# ── 7. MIXED DATA TYPES ─────────────────────────────────────────────────────
print("\n[7] MIXED DATA TYPES IN STRING COLUMNS")
for col in str_cols:
    numeric_looking = df[col].dropna().str.match(r'^\s*-?\d+\.?\d*\s*$').sum()
    total = df[col].notna().sum()
    if 0 < numeric_looking < total:
        print(f"    ✗ '{col}': {numeric_looking:,} cells look numeric inside a text column")
        issues[col].append(f"{numeric_looking} cells appear numeric")
    else:
        print(f"    ✓ '{col}': consistent type")

# ── 8. NEAR-DUPLICATE VALUES (within string cols) ───────────────────────────
print("\n[8] NEAR-DUPLICATES (values differing only by spaces/case)")
for col in str_cols:
    normalized = df[col].str.strip().str.lower()
    dupe_mask = normalized.duplicated(keep=False)
    raw_dupes = df[col][dupe_mask & df[col].str.strip().str.lower().ne(df[col])]
    count = raw_dupes.nunique()
    if count:
        print(f"    ✗ '{col}': {count:,} near-duplicate values (same after normalizing)")
        issues[col].append(f"{count} near-duplicates")
    else:
        print(f"    ✓ '{col}': no near-duplicates detected")

# ── SUMMARY ─────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  SUMMARY OF ISSUES")
print("=" * 60)
if not issues:
    print("  ✓ Dataset looks clean!")
else:
    all_issues = 0
    for key, vals in issues.items():
        label = "Dataset" if key == "_dataset" else f"Column '{key}'"
        for v in vals:
            print(f"  • {label}: {v}")
            all_issues += 1
    print(f"\n  Total issue types found: {all_issues}")
print("=" * 60)

  CSV QUALITY REPORT — restaurant_ratings_fixed.csv
  3,068 rows × 5 columns

[1] MISSING VALUES
    ✓ No missing values

[2] DUPLICATE ROWS
    ✓ No duplicate rows

[3] LEADING / TRAILING WHITESPACE
    ✓ 'restaurant_name': clean
    ✓ 'location': clean
    ✓ 'rating_text': clean

[4] INCONSISTENT CASING
    ✓ 'restaurant_name': consistent
    ✓ 'location': consistent
    ✓ 'rating_text': consistent

[5] ENCODING / GARBLED TEXT
    ✗ 'restaurant_name': 21 cells with encoding issues  e.g. 'CafãÂÃÂÃÂÃÂÃÂÃÂÃÂÃÂ© Down The Alley'
    ✓ 'location': clean
    ✗ 'rating_text': 1,003 cells with encoding issues  e.g. '"[('Rated 4.0', 'RATED\n  A beautiful place to dine in.The interiors take you back to the Mughal era. The lightings are just perfect.We went there on the occasion of Christmas and so they had only limited items available. But the taste and service was not compromised at all.The only complaint is that the breads could have been better.Would surely like to come here a

In [ ]:
def fix_encoding(x):
    if isinstance(x, str):
        try:
            return x.encode("latin1").decode("utf-8")
        except:
            return x
    return x


df["restaurant_name"] = df["restaurant_name"].apply(fix_encoding)
df["rating_text"] = df["rating_text"].apply(fix_encoding)

In [ ]:
df = df.drop(columns=["rating_text"], errors="ignore")

In [ ]:
df["restaurant_name"] = (
    df["restaurant_name"]
    .str.lower()
    .str.strip()
)

df["location"] = (
    df["location"]
    .str.lower()
    .str.strip()
)

In [ ]:
df = df.drop_duplicates()

In [ ]:
df = df[~df["restaurant_name"].str.isnumeric()]

In [ ]:
df = df[(df["rating"] >= 2.5) & (df["rating"] <= 5)]

In [ ]:
q1 = df["votes"].quantile(0.25)
q3 = df["votes"].quantile(0.75)

iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

df = df[(df["votes"] >= lower) & (df["votes"] <= upper)]

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
df.to_csv("restaurant_cleaned_final_1.csv", index=False)

In [ ]:
import pandas as pd
import unicodedata
import re
from collections import defaultdict

# ── CONFIG ──────────────────────────────────────────────────────────────────
FILE = "restaurant_cleaned_final_1.csv"  # ← change to your filename
# ────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(FILE)
issues = defaultdict(list)

print("=" * 60)
print(f"  CSV QUALITY REPORT — {FILE}")
print(f"  {df.shape[0]:,} rows × {df.shape[1]} columns")
print("=" * 60)

# ── 1. MISSING VALUES ────────────────────────────────────────────────────────
print("\n[1] MISSING VALUES")
nulls = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(2)
has_nulls = False
for col in df.columns:
    if nulls[col] > 0:
        print(f"    ✗ '{col}': {nulls[col]:,} nulls ({null_pct[col]}%)")
        issues[col].append(f"{nulls[col]} missing values")
        has_nulls = True
if not has_nulls:
    print("    ✓ No missing values")

# ── 2. DUPLICATE ROWS ────────────────────────────────────────────────────────
print("\n[2] DUPLICATE ROWS")
dupe_count = df.duplicated().sum()
if dupe_count:
    print(f"    ✗ {dupe_count:,} fully duplicate rows found")
    issues["_dataset"].append(f"{dupe_count} duplicate rows")
else:
    print("    ✓ No duplicate rows")

# ── 3. WHITESPACE ISSUES (string columns) ───────────────────────────────────
print("\n[3] LEADING / TRAILING WHITESPACE")
str_cols = df.select_dtypes(include="object").columns
for col in str_cols:
    ws = df[col].str.strip().ne(df[col]).sum()
    if ws:
        print(f"    ✗ '{col}': {ws:,} cells have extra whitespace")
        issues[col].append(f"{ws} whitespace issues")
    else:
        print(f"    ✓ '{col}': clean")

# ── 4. INCONSISTENT CASING ──────────────────────────────────────────────────
print("\n[4] INCONSISTENT CASING")
for col in str_cols:
    unique_raw = df[col].str.strip().dropna().unique()
    groups = defaultdict(list)
    for v in unique_raw:
        groups[v.lower()].append(v)
    mixed = {k: v for k, v in groups.items() if len(v) > 1}
    if mixed:
        examples = list(mixed.values())[:3]
        ex_str = " | ".join([str(e) for e in examples])
        print(f"    ✗ '{col}': {len(mixed):,} values with inconsistent casing  e.g. {ex_str}")
        issues[col].append(f"{len(mixed)} casing inconsistencies")
    else:
        print(f"    ✓ '{col}': consistent")

# ── 5. ENCODING ISSUES ──────────────────────────────────────────────────────
print("\n[5] ENCODING / GARBLED TEXT")
def has_encoding_issue(text):
    if not isinstance(text, str):
        return False
    # Flag classic mojibake multi-byte garbage sequences
    return bool(re.search(r'[ãâïÃÂ]{2,}|[\x80-\x9f]|â€|Â·|Ã©|Ã¨|Ã¼', text))

for col in str_cols:
    bad = df[col].apply(has_encoding_issue).sum()
    if bad:
        example = df[col][df[col].apply(has_encoding_issue)].iloc[0]
        print(f"    ✗ '{col}': {bad:,} cells with encoding issues  e.g. '{example}'")
        issues[col].append(f"{bad} encoding issues")
    else:
        print(f"    ✓ '{col}': clean")

# ── 6. NUMERIC OUTLIERS ─────────────────────────────────────────────────────
print("\n[6] NUMERIC OUTLIERS (IQR method)")
num_cols = df.select_dtypes(include="number").columns
for col in num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    if len(outliers):
        print(f"    ✗ '{col}': {len(outliers):,} outliers  (expected {lower:.1f}–{upper:.1f}, got min={df[col].min()}, max={df[col].max()})")
        issues[col].append(f"{len(outliers)} outliers")
    else:
        print(f"    ✓ '{col}': no outliers")

# ── 7. MIXED DATA TYPES ─────────────────────────────────────────────────────
print("\n[7] MIXED DATA TYPES IN STRING COLUMNS")
for col in str_cols:
    numeric_looking = df[col].dropna().str.match(r'^\s*-?\d+\.?\d*\s*$').sum()
    total = df[col].notna().sum()
    if 0 < numeric_looking < total:
        print(f"    ✗ '{col}': {numeric_looking:,} cells look numeric inside a text column")
        issues[col].append(f"{numeric_looking} cells appear numeric")
    else:
        print(f"    ✓ '{col}': consistent type")

# ── 8. NEAR-DUPLICATE VALUES (within string cols) ───────────────────────────
print("\n[8] NEAR-DUPLICATES (values differing only by spaces/case)")
for col in str_cols:
    normalized = df[col].str.strip().str.lower()
    dupe_mask = normalized.duplicated(keep=False)
    raw_dupes = df[col][dupe_mask & df[col].str.strip().str.lower().ne(df[col])]
    count = raw_dupes.nunique()
    if count:
        print(f"    ✗ '{col}': {count:,} near-duplicate values (same after normalizing)")
        issues[col].append(f"{count} near-duplicates")
    else:
        print(f"    ✓ '{col}': no near-duplicates detected")

# ── SUMMARY ─────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  SUMMARY OF ISSUES")
print("=" * 60)
if not issues:
    print("  ✓ Dataset looks clean!")
else:
    all_issues = 0
    for key, vals in issues.items():
        label = "Dataset" if key == "_dataset" else f"Column '{key}'"
        for v in vals:
            print(f"  • {label}: {v}")
            all_issues += 1
    print(f"\n  Total issue types found: {all_issues}")
print("=" * 60)

  CSV QUALITY REPORT — restaurant_cleaned_final_1.csv
  2,337 rows × 4 columns

[1] MISSING VALUES
    ✓ No missing values

[2] DUPLICATE ROWS
    ✓ No duplicate rows

[3] LEADING / TRAILING WHITESPACE
    ✓ 'restaurant_name': clean
    ✓ 'location': clean

[4] INCONSISTENT CASING
    ✓ 'restaurant_name': consistent
    ✓ 'location': consistent

[5] ENCODING / GARBLED TEXT
    ✗ 'restaurant_name': 16 cells with encoding issues  e.g. 'cafãâãâãâãâãâãâãâãâ© down the alley'
    ✓ 'location': clean

[6] NUMERIC OUTLIERS (IQR method)
    ✓ 'rating': no outliers
    ✗ 'votes': 166 outliers  (expected -154.0–302.0, got min=0.0, max=486.0)

[7] MIXED DATA TYPES IN STRING COLUMNS
    ✓ 'restaurant_name': consistent type
    ✓ 'location': consistent type

[8] NEAR-DUPLICATES (values differing only by spaces/case)
    ✓ 'restaurant_name': no near-duplicates detected
    ✓ 'location': no near-duplicates detected

  SUMMARY OF ISSUES
  • Column 'restaurant_name': 16 encoding issues
  

In [ ]:
def fix_encoding(x):
    if isinstance(x, str):
        try:
            return x.encode("latin1", errors="ignore").decode("utf-8", errors="ignore")
        except:
            return x
    return x

df["restaurant_name"] = df["restaurant_name"].apply(fix_encoding)

In [ ]:
df = df[~df["restaurant_name"].str.contains("ã|â|�", na=False)]

In [ ]:
q1 = df["votes"].quantile(0.25)
q3 = df["votes"].quantile(0.75)

iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

df = df[(df["votes"] >= lower) & (df["votes"] <= upper)]

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
df.to_csv("restaurant_cleaned_final_2.csv", index=False)

In [ ]:
import pandas as pd
import unicodedata
import re
from collections import defaultdict

# ── CONFIG ──────────────────────────────────────────────────────────────────
FILE = "restaurant_cleaned_final_2.csv"  # ← change to your filename
# ────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(FILE)
issues = defaultdict(list)

print("=" * 60)
print(f"  CSV QUALITY REPORT — {FILE}")
print(f"  {df.shape[0]:,} rows × {df.shape[1]} columns")
print("=" * 60)

# ── 1. MISSING VALUES ────────────────────────────────────────────────────────
print("\n[1] MISSING VALUES")
nulls = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(2)
has_nulls = False
for col in df.columns:
    if nulls[col] > 0:
        print(f"    ✗ '{col}': {nulls[col]:,} nulls ({null_pct[col]}%)")
        issues[col].append(f"{nulls[col]} missing values")
        has_nulls = True
if not has_nulls:
    print("    ✓ No missing values")

# ── 2. DUPLICATE ROWS ────────────────────────────────────────────────────────
print("\n[2] DUPLICATE ROWS")
dupe_count = df.duplicated().sum()
if dupe_count:
    print(f"    ✗ {dupe_count:,} fully duplicate rows found")
    issues["_dataset"].append(f"{dupe_count} duplicate rows")
else:
    print("    ✓ No duplicate rows")

# ── 3. WHITESPACE ISSUES (string columns) ───────────────────────────────────
print("\n[3] LEADING / TRAILING WHITESPACE")
str_cols = df.select_dtypes(include="object").columns
for col in str_cols:
    ws = df[col].str.strip().ne(df[col]).sum()
    if ws:
        print(f"    ✗ '{col}': {ws:,} cells have extra whitespace")
        issues[col].append(f"{ws} whitespace issues")
    else:
        print(f"    ✓ '{col}': clean")

# ── 4. INCONSISTENT CASING ──────────────────────────────────────────────────
print("\n[4] INCONSISTENT CASING")
for col in str_cols:
    unique_raw = df[col].str.strip().dropna().unique()
    groups = defaultdict(list)
    for v in unique_raw:
        groups[v.lower()].append(v)
    mixed = {k: v for k, v in groups.items() if len(v) > 1}
    if mixed:
        examples = list(mixed.values())[:3]
        ex_str = " | ".join([str(e) for e in examples])
        print(f"    ✗ '{col}': {len(mixed):,} values with inconsistent casing  e.g. {ex_str}")
        issues[col].append(f"{len(mixed)} casing inconsistencies")
    else:
        print(f"    ✓ '{col}': consistent")

# ── 5. ENCODING ISSUES ──────────────────────────────────────────────────────
print("\n[5] ENCODING / GARBLED TEXT")
def has_encoding_issue(text):
    if not isinstance(text, str):
        return False
    # Flag classic mojibake multi-byte garbage sequences
    return bool(re.search(r'[ãâïÃÂ]{2,}|[\x80-\x9f]|â€|Â·|Ã©|Ã¨|Ã¼', text))

for col in str_cols:
    bad = df[col].apply(has_encoding_issue).sum()
    if bad:
        example = df[col][df[col].apply(has_encoding_issue)].iloc[0]
        print(f"    ✗ '{col}': {bad:,} cells with encoding issues  e.g. '{example}'")
        issues[col].append(f"{bad} encoding issues")
    else:
        print(f"    ✓ '{col}': clean")

# ── 6. NUMERIC OUTLIERS ─────────────────────────────────────────────────────
print("\n[6] NUMERIC OUTLIERS (IQR method)")
num_cols = df.select_dtypes(include="number").columns
for col in num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    if len(outliers):
        print(f"    ✗ '{col}': {len(outliers):,} outliers  (expected {lower:.1f}–{upper:.1f}, got min={df[col].min()}, max={df[col].max()})")
        issues[col].append(f"{len(outliers)} outliers")
    else:
        print(f"    ✓ '{col}': no outliers")

# ── 7. MIXED DATA TYPES ─────────────────────────────────────────────────────
print("\n[7] MIXED DATA TYPES IN STRING COLUMNS")
for col in str_cols:
    numeric_looking = df[col].dropna().str.match(r'^\s*-?\d+\.?\d*\s*$').sum()
    total = df[col].notna().sum()
    if 0 < numeric_looking < total:
        print(f"    ✗ '{col}': {numeric_looking:,} cells look numeric inside a text column")
        issues[col].append(f"{numeric_looking} cells appear numeric")
    else:
        print(f"    ✓ '{col}': consistent type")

# ── 8. NEAR-DUPLICATE VALUES (within string cols) ───────────────────────────
print("\n[8] NEAR-DUPLICATES (values differing only by spaces/case)")
for col in str_cols:
    normalized = df[col].str.strip().str.lower()
    dupe_mask = normalized.duplicated(keep=False)
    raw_dupes = df[col][dupe_mask & df[col].str.strip().str.lower().ne(df[col])]
    count = raw_dupes.nunique()
    if count:
        print(f"    ✗ '{col}': {count:,} near-duplicate values (same after normalizing)")
        issues[col].append(f"{count} near-duplicates")
    else:
        print(f"    ✓ '{col}': no near-duplicates detected")

# ── SUMMARY ─────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  SUMMARY OF ISSUES")
print("=" * 60)
if not issues:
    print("  ✓ Dataset looks clean!")
else:
    all_issues = 0
    for key, vals in issues.items():
        label = "Dataset" if key == "_dataset" else f"Column '{key}'"
        for v in vals:
            print(f"  • {label}: {v}")
            all_issues += 1
    print(f"\n  Total issue types found: {all_issues}")
print("=" * 60)

  CSV QUALITY REPORT — restaurant_cleaned_final_2.csv
  2,171 rows × 4 columns

[1] MISSING VALUES
    ✓ No missing values

[2] DUPLICATE ROWS
    ✓ No duplicate rows

[3] LEADING / TRAILING WHITESPACE
    ✓ 'restaurant_name': clean
    ✓ 'location': clean

[4] INCONSISTENT CASING
    ✓ 'restaurant_name': consistent
    ✓ 'location': consistent

[5] ENCODING / GARBLED TEXT
    ✓ 'restaurant_name': clean
    ✓ 'location': clean

[6] NUMERIC OUTLIERS (IQR method)
    ✗ 'rating': 5 outliers  (expected 2.5–4.5, got min=2.5, max=4.7)
    ✗ 'votes': 89 outliers  (expected -119.0–241.0, got min=0.0, max=302.0)

[7] MIXED DATA TYPES IN STRING COLUMNS
    ✓ 'restaurant_name': consistent type
    ✓ 'location': consistent type

[8] NEAR-DUPLICATES (values differing only by spaces/case)
    ✓ 'restaurant_name': no near-duplicates detected
    ✓ 'location': no near-duplicates detected

  SUMMARY OF ISSUES
  • Column 'rating': 5 outliers
  • Column 'votes': 89 outliers

  Total issue types found: 2


In [ ]:
df["rating"] = df["rating"].clip(2.5, 4.5)

In [ ]:
q1 = df["votes"].quantile(0.25)
q3 = df["votes"].quantile(0.75)

iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

df["votes"] = df["votes"].clip(lower, upper)

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
df.to_csv("restaurant_cleaned_FINAL.csv", index=False)

In [ ]:
print("Dataset A columns:", df_v1.columns.tolist())
print("Dataset B columns:", df_v2.columns.tolist())

Dataset A columns: ['restaurant_name', 'location', 'rating', 'votes']
Dataset B columns: ['restaurant_name', 'location', 'cuisines', 'cost', 'online_delivery', 'book_table']


In [ ]:
# ============================================================
#  🍽️  ZOMATO UNCOVERED — FULL ANALYSIS (FIXED)
#  Files: restaurant_cleaned_FINAL.csv + restaurant_final_v3_perfect.csv
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# ── LOAD DATA ────────────────────────────────────────────────
df_ratings = pd.read_csv('restaurant_cleaned_FINAL.csv',
                         on_bad_lines='skip', quoting=3, engine='python')
df_info    = pd.read_csv('restaurant_final_v3_perfect.csv',
                         on_bad_lines='skip', quoting=3, engine='python')

# Clean info file
df_info.columns = df_info.columns.str.strip().str.lower()
df_info['restaurant_name'] = df_info['restaurant_name'].str.strip().str.title()
df_info['location']        = df_info['location'].str.strip().str.title()
df_info['cuisines']        = df_info['cuisines'].str.strip().str.title()
df_info['cost']            = pd.to_numeric(df_info['cost'], errors='coerce')
df_info['online_delivery'] = pd.to_numeric(df_info['online_delivery'], errors='coerce').fillna(0).astype(int)
df_info['book_table']      = pd.to_numeric(df_info['book_table'], errors='coerce').fillna(0).astype(int)
df_info = df_info.dropna(subset=['cost', 'cuisines', 'location'])

# Clean ratings file
df_ratings.columns = df_ratings.columns.str.strip().str.lower()
df_ratings['restaurant_name'] = df_ratings['restaurant_name'].str.strip().str.title()
df_ratings['location']        = df_ratings['location'].str.strip().str.title()
df_ratings['rating'] = pd.to_numeric(df_ratings['rating'], errors='coerce')
df_ratings['votes']  = pd.to_numeric(df_ratings['votes'],  errors='coerce')

# Merge ratings into info
df_all = df_info.merge(
    df_ratings[['restaurant_name', 'location', 'rating', 'votes']],
    on=['restaurant_name', 'location'], how='left'
).drop_duplicates(subset=['restaurant_name', 'location', 'cuisines'])

has_rating = df_all['rating'].notna().sum() > 100

print(f"Info:    {len(df_info):,} rows")
print(f"Ratings: {len(df_ratings):,} rows")
print(f"Merged:  {len(df_all):,} rows | with rating: {df_all['rating'].notna().sum():,}")

# ── GLOBAL STYLE ─────────────────────────────────────────────
BG     = '#0d0d0d'
CARD   = '#161616'
CARD2  = '#1e1e1e'
A      = '#ff4d00'
B      = '#00c9a7'
GOLD   = '#ffc300'
PINK   = '#ff6b9d'
PURPLE = '#a855f7'
BLUE   = '#38bdf8'
WHITE  = '#efefef'
MUTED  = '#666666'
GRID   = '#222222'
PAL    = [A, B, GOLD, PINK, PURPLE, BLUE, '#34d399', '#fb923c', '#e879f9', '#60a5fa']

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': CARD,
    'axes.edgecolor': '#2a2a2a', 'axes.labelcolor': WHITE,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'text.color': WHITE, 'grid.color': GRID,
    'grid.linestyle': '--', 'font.family': 'monospace',
    'axes.spines.top': False, 'axes.spines.right': False,
})

def save(fig, name):
    fig.savefig(name, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close(fig)
    print(f"   ✓ {name}")

def note(fig, y, text, color=MUTED):
    fig.text(0.5, y, text, ha='center', fontsize=9, color=color, style='italic',
             bbox=dict(boxstyle='round,pad=0.4', facecolor=BG, edgecolor=color, alpha=0.7))


# ════════════════════════════════════════════════════════════
#  VIZ 1 — DATASET OVERVIEW
# ════════════════════════════════════════════════════════════
print("\n[1] Dataset Overview...")

fig, axes = plt.subplots(2, 3, figsize=(20, 11), facecolor=BG)
fig.suptitle('DATASET OVERVIEW — Restaurant Info + Ratings Combined',
             fontsize=17, fontweight='bold', color=WHITE, y=0.98)

# Top cuisines
ax = axes[0][0]
top = df_info['cuisines'].value_counts().head(10)
ax.barh(range(len(top)), top.values,
        color=[PAL[i % len(PAL)] for i in range(len(top))], edgecolor='none', height=0.65)
ax.set_yticks(range(len(top))); ax.set_yticklabels(top.index, fontsize=9)
ax.invert_yaxis()
ax.set_title('Top 10 Cuisines', fontsize=11, color=B, pad=10)
ax.grid(axis='x', alpha=0.3)

# Top locations
ax = axes[0][1]
top_loc = df_info['location'].value_counts().head(10)
ax.barh(range(len(top_loc)), top_loc.values,
        color=[PAL[i % len(PAL)] for i in range(len(top_loc))], edgecolor='none', height=0.65)
ax.set_yticks(range(len(top_loc))); ax.set_yticklabels(top_loc.index, fontsize=9)
ax.invert_yaxis()
ax.set_title('Top 10 Locations', fontsize=11, color=A, pad=10)
ax.grid(axis='x', alpha=0.3)

# Rating distribution
ax = axes[0][2]
if has_rating:
    rated = df_all[df_all['rating'].notna()]
    ax.hist(rated['rating'], bins=20, color=GOLD, edgecolor=BG, alpha=0.85)
    ax.axvline(rated['rating'].mean(),   color=A, linestyle='--', label=f"Mean: {rated['rating'].mean():.2f}")
    ax.axvline(rated['rating'].median(), color=B, linestyle='--', label=f"Median: {rated['rating'].median():.2f}")
    ax.set_title(f'Rating Distribution\n({len(rated):,} restaurants)', fontsize=11, color=GOLD, pad=10)
    ax.legend(fontsize=9, frameon=False)
ax.set_xlabel('Rating', fontsize=10); ax.grid(alpha=0.3)

# Cost distribution
ax = axes[1][0]
ax.hist(df_info['cost'].clip(0, 2000), bins=40, color=A, edgecolor='none', alpha=0.8)
ax.axvline(df_info['cost'].median(), color=GOLD, linestyle='--',
           label=f"Median ₹{int(df_info['cost'].median())}")
ax.set_xlabel('Cost for Two (₹)', fontsize=10)
ax.set_title('Cost Distribution', fontsize=11, color=WHITE, pad=10)
ax.legend(fontsize=9, frameon=False); ax.grid(alpha=0.3)

# Feature adoption
ax = axes[1][1]
vals = [df_info['online_delivery'].mean()*100, df_info['book_table'].mean()*100]
bars = ax.bar(['Online Delivery', 'Table Booking'], vals, color=[A, B], edgecolor='none', width=0.5)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.5, f'{v:.1f}%',
            ha='center', fontsize=12, color=WHITE, fontweight='bold')
ax.set_ylabel('% of Restaurants', fontsize=10)
ax.set_title('Feature Adoption', fontsize=11, color=WHITE, pad=10)
ax.grid(axis='y', alpha=0.3)

# Votes distribution
ax = axes[1][2]
if has_rating:
    voted = df_all[df_all['votes'].notna() & (df_all['votes'] > 0)]
    ax.hist(voted['votes'].clip(0, 2000), bins=30, color=PURPLE, edgecolor='none', alpha=0.85)
    ax.axvline(voted['votes'].median(), color=GOLD, linestyle='--',
               label=f"Median: {int(voted['votes'].median())} votes")
    ax.set_title(f'Votes Distribution\n({len(voted):,} restaurants)', fontsize=11, color=PURPLE, pad=10)
    ax.legend(fontsize=9, frameon=False)
ax.set_xlabel('Votes', fontsize=10); ax.grid(alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.97])
save(fig, 'viz1_dataset_overview.png')


# ════════════════════════════════════════════════════════════
#  VIZ 2 — CUISINE DEEP DIVE
# ════════════════════════════════════════════════════════════
print("[2] Cuisine Deep Dive...")

fig, axes = plt.subplots(1, 3, figsize=(22, 9), facecolor=BG)
fig.suptitle('CUISINE DEEP DIVE — Supply, Cost & Market Share',
             fontsize=17, fontweight='bold', color=A, y=0.98)

top20 = df_all['cuisines'].value_counts().head(20)

# Volume bar
ax = axes[0]
ax.barh(range(20), top20.values,
        color=[PAL[i % len(PAL)] for i in range(20)], edgecolor='none', height=0.72)
ax.set_yticks(range(20)); ax.set_yticklabels(top20.index, fontsize=9)
ax.invert_yaxis()
ax.set_title('TOP 20 CUISINES BY VOLUME', fontsize=11, color=WHITE, pad=12)
ax.set_xlabel('Total Restaurants', fontsize=10)
ax.grid(axis='x', alpha=0.3)

# Avg cost per cuisine
ax2 = axes[1]
top15_cost = (df_all[df_all['cuisines'].isin(top20.index[:15])]
              .groupby('cuisines')['cost'].mean().sort_values(ascending=True))
ax2.barh(range(len(top15_cost)), top15_cost.values,
         color=[PAL[list(top20.index).index(c) % len(PAL)] for c in top15_cost.index],
         edgecolor='none', height=0.72)
ax2.set_yticks(range(len(top15_cost))); ax2.set_yticklabels(top15_cost.index, fontsize=9)
ax2.set_title('AVG COST BY CUISINE (₹)', fontsize=11, color=WHITE, pad=12)
ax2.set_xlabel('Avg Cost (₹)', fontsize=10)
ax2.grid(axis='x', alpha=0.3)
for i, v in enumerate(top15_cost.values):
    ax2.text(v+5, i, f'₹{int(v)}', va='center', fontsize=8, color=MUTED)

# Donut — market share
ax3 = axes[2]
top8 = df_all['cuisines'].value_counts().head(8)
other = len(df_all) - top8.sum()
vals  = list(top8.values) + [other]
lbls  = list(top8.index)  + ['Others']
clrs  = PAL[:8] + [MUTED]
wedges, _, autotexts = ax3.pie(
    vals, labels=None, colors=clrs, startangle=90,
    autopct='%1.1f%%', pctdistance=0.78,
    wedgeprops=dict(edgecolor=BG, linewidth=2.5, width=0.55))
for at in autotexts:
    at.set_fontsize(8); at.set_color(WHITE)
ax3.legend(wedges, lbls, loc='lower center', ncol=3, fontsize=8,
           frameon=False, bbox_to_anchor=(0.5, -0.12))
ax3.set_title('MARKET SHARE (Top 8)', fontsize=11, color=WHITE, pad=12)

note(fig, 0.02, f"North Indian = {top8.iloc[0]/len(df_all)*100:.1f}% of all restaurants. "
     "Massive saturation — niche cuisines have room to grow.", A)
plt.tight_layout(rect=[0, 0.06, 1, 0.96])
save(fig, 'viz2_cuisine_deepdive.png')


# ════════════════════════════════════════════════════════════
#  VIZ 3 — PRICE INTELLIGENCE
# ════════════════════════════════════════════════════════════
print("[3] Price Intelligence...")

fig, axes = plt.subplots(2, 2, figsize=(18, 13), facecolor=BG)
fig.suptitle('PRICE INTELLIGENCE — Where Does Value Actually Live?',
             fontsize=17, fontweight='bold', color=GOLD, y=0.98)

df3 = df_all[df_all['cost'].between(50, 3000)].copy()

# Price tier line by dataset equivalent (info vs rated subset)
ax = axes[0][0]
for d, col, name in [(df_info, A, 'All Restaurants'),
                      (df_all[df_all['rating'].notna()], B, 'Rated Restaurants')]:
    d2 = d[d['cost'].between(50,3000)].copy()
    d2['tier'] = pd.cut(d2['cost'], bins=[0,200,400,700,1200,10000],
                        labels=['Budget','Economy','Mid','Premium','Luxury'])
    counts = d2['tier'].value_counts().sort_index()
    ax.plot(counts.index, counts.values, marker='o', color=col,
            linewidth=2.5, markersize=8, label=name)
    ax.fill_between(range(len(counts)), counts.values, alpha=0.1, color=col)
ax.set_title('PRICE TIER DISTRIBUTION', fontsize=11, color=WHITE, pad=12)
ax.legend(fontsize=9, frameon=False); ax.set_ylabel('Count', fontsize=10)
ax.grid(alpha=0.3)

# Cost boxplot per top cuisine
ax2 = axes[0][1]
top10_c = df_all['cuisines'].value_counts().head(10).index
df3b = df_all[df_all['cuisines'].isin(top10_c) & df_all['cost'].between(50, 2000)]
cuisine_order = df3b.groupby('cuisines')['cost'].median().sort_values().index.tolist()
bp_data = [df3b[df3b['cuisines'] == c]['cost'].values for c in cuisine_order]
bp = ax2.boxplot(bp_data, vert=False, patch_artist=True,
                 medianprops=dict(color=WHITE, linewidth=2),
                 whiskerprops=dict(color=MUTED), capprops=dict(color=MUTED),
                 flierprops=dict(marker='o', markerfacecolor=MUTED, markersize=3, alpha=0.3))
for patch, color in zip(bp['boxes'], PAL):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax2.set_yticks(range(1, len(cuisine_order)+1))
ax2.set_yticklabels(cuisine_order, fontsize=9)
ax2.set_xlabel('Cost for Two (₹)', fontsize=10)
ax2.set_title('COST SPREAD BY CUISINE', fontsize=11, color=WHITE, pad=12)
ax2.grid(axis='x', alpha=0.3)

# Avg cost by feature combo
ax3 = axes[1][0]
combos = [(0,0,'Neither',MUTED),(1,0,'Delivery Only',A),(0,1,'Booking Only',B),(1,1,'Both',GOLD)]
labels_c, means_c, counts_c, clrs_c = [], [], [], []
for d, b, lbl, clr in combos:
    sub = df_all[(df_all['online_delivery']==d) & (df_all['book_table']==b)]['cost']
    if len(sub) > 0:
        labels_c.append(lbl); means_c.append(sub.mean())
        counts_c.append(len(sub)); clrs_c.append(clr)
bars3 = ax3.bar(labels_c, means_c, color=clrs_c, edgecolor='none', width=0.55)
for bar, v, cnt in zip(bars3, means_c, counts_c):
    ax3.text(bar.get_x()+bar.get_width()/2, v+8,
             f'₹{int(v)}\nn={cnt:,}', ha='center', fontsize=10,
             color=WHITE, fontweight='bold', linespacing=1.5)
ax3.set_ylabel('Avg Cost (₹)', fontsize=10)
ax3.set_title('AVG PRICE BY FEATURE COMBO', fontsize=11, color=WHITE, pad=12)
ax3.grid(axis='y', alpha=0.3)

# Cost heatmap location x cuisine
ax4 = axes[1][1]
top8_loc = df_all['location'].value_counts().head(8).index
top8_cui = df_all['cuisines'].value_counts().head(8).index
pivot = (df_all[df_all['location'].isin(top8_loc) & df_all['cuisines'].isin(top8_cui)]
         .groupby(['location','cuisines'])['cost'].mean().unstack().fillna(0))
sns.heatmap(pivot, ax=ax4, annot=True, fmt='.0f',
            cmap=LinearSegmentedColormap.from_list('heat', [CARD, GOLD, A]),
            linewidths=1, linecolor=BG, annot_kws={'size':8,'color':BG},
            cbar_kws={'shrink':0.8})
ax4.set_title('AVG COST: Location x Cuisine', fontsize=11, color=WHITE, pad=12)
ax4.tick_params(axis='x', rotation=35, labelsize=8)
ax4.tick_params(axis='y', rotation=0, labelsize=8)

note(fig, 0.01, "Restaurants with table booking charge ~2x more on average — a strong premium signal.", GOLD)
plt.tight_layout(rect=[0, 0.04, 1, 0.96])
save(fig, 'viz3_price_intelligence.png')


# ════════════════════════════════════════════════════════════
#  VIZ 4 — RATINGS ANALYSIS
# ════════════════════════════════════════════════════════════
print("[4] Ratings Analysis...")

fig, axes = plt.subplots(2, 2, figsize=(18, 13), facecolor=BG)
fig.suptitle('RATINGS ANALYSIS — What Gets Loved vs Ignored',
             fontsize=17, fontweight='bold', color=GOLD, y=0.98)

if has_rating:
    rated = df_all[df_all['rating'].notna()].copy()

    # Avg rating by cuisine
    ax = axes[0][0]
    top12_cui = df_all['cuisines'].value_counts().head(12).index
    rat_cui = (rated[rated['cuisines'].isin(top12_cui)]
               .groupby('cuisines')['rating'].mean().sort_values(ascending=True))
    ax.barh(range(len(rat_cui)), rat_cui.values,
            color=[PAL[i%len(PAL)] for i in range(len(rat_cui))], edgecolor='none', height=0.65)
    ax.set_yticks(range(len(rat_cui))); ax.set_yticklabels(rat_cui.index, fontsize=9)
    ax.axvline(rated['rating'].mean(), color=WHITE, linestyle='--', alpha=0.5, label='Overall avg')
    ax.set_xlabel('Avg Rating', fontsize=10)
    ax.set_title('AVG RATING BY CUISINE', fontsize=11, color=WHITE, pad=12)
    ax.set_xlim(0, 5); ax.legend(fontsize=8, frameon=False)
    ax.grid(axis='x', alpha=0.3)
    for i, v in enumerate(rat_cui.values):
        ax.text(v+0.02, i, f'{v:.2f}', va='center', fontsize=8, color=MUTED)

    # Avg rating by location
    ax2 = axes[0][1]
    top12_loc = df_all['location'].value_counts().head(12).index
    rat_loc = (rated[rated['location'].isin(top12_loc)]
               .groupby('location')['rating'].mean().sort_values(ascending=True))
    ax2.barh(range(len(rat_loc)), rat_loc.values,
             color=[PAL[i%len(PAL)] for i in range(len(rat_loc))], edgecolor='none', height=0.65)
    ax2.set_yticks(range(len(rat_loc))); ax2.set_yticklabels(rat_loc.index, fontsize=9)
    ax2.axvline(rated['rating'].mean(), color=WHITE, linestyle='--', alpha=0.5)
    ax2.set_xlabel('Avg Rating', fontsize=10)
    ax2.set_title('AVG RATING BY LOCATION', fontsize=11, color=WHITE, pad=12)
    ax2.set_xlim(0, 5); ax2.grid(axis='x', alpha=0.3)

    # Rating vs cost scatter
    ax3 = axes[1][0]
    sample = rated[rated['cost'].between(50, 2500)].sample(min(1500, len(rated)), random_state=42)
    sc = ax3.scatter(sample['cost'], sample['rating'], alpha=0.35, s=15,
                     c=sample['cost'],
                     cmap=LinearSegmentedColormap.from_list('rg',[B, GOLD, A]),
                     edgecolors='none')
    z = np.polyfit(sample['cost'], sample['rating'], 1)
    xs = np.linspace(sample['cost'].min(), sample['cost'].max(), 100)
    ax3.plot(xs, np.poly1d(z)(xs), color=WHITE, linewidth=2, linestyle='--', label='Trend')
    ax3.set_xlabel('Cost for Two (₹)', fontsize=10)
    ax3.set_ylabel('Rating', fontsize=10)
    ax3.set_title('COST vs RATING', fontsize=11, color=WHITE, pad=12)
    ax3.legend(fontsize=9, frameon=False); ax3.grid(alpha=0.3)

    # Rating by delivery/booking
    ax4 = axes[1][1]
    for feat, label, color, offset in [('online_delivery','Online Delivery',A,-0.2),
                                        ('book_table','Table Booking',B,0.2)]:
        g0 = rated[rated[feat]==0]['rating'].dropna()
        g1 = rated[rated[feat]==1]['rating'].dropna()
        parts = ax4.violinplot([g0, g1],
                               positions=[0+offset, 1+offset],
                               showmedians=True, widths=0.35)
        for pc in parts['bodies']:
            pc.set_facecolor(color); pc.set_alpha(0.5)
        parts['cmedians'].set_color(WHITE)
        for key in ['cbars','cmins','cmaxes']:
            parts[key].set_color(MUTED)
    ax4.set_xticks([0, 1])
    ax4.set_xticklabels(['No Feature', 'Has Feature'], fontsize=10)
    ax4.set_ylabel('Rating', fontsize=10)
    ax4.set_title('RATING: Feature vs No Feature\n(Orange=Delivery, Teal=Booking)',
                  fontsize=11, color=WHITE, pad=12)
    ax4.grid(axis='y', alpha=0.3)
    legend_els = [mpatches.Patch(color=A, label='Delivery'), mpatches.Patch(color=B, label='Booking')]
    ax4.legend(handles=legend_els, fontsize=9, frameon=False)

note(fig, 0.01, "Higher-rated restaurants tend to cluster in the ₹400–₹800 range — "
     "not the cheapest, not the priciest.", GOLD)
plt.tight_layout(rect=[0, 0.04, 1, 0.96])
save(fig, 'viz4_ratings_analysis.png')


# ════════════════════════════════════════════════════════════
#  VIZ 5 — LOCATION POWER MAP
# ════════════════════════════════════════════════════════════
print("[5] Location Power Map...")

fig, axes = plt.subplots(1, 3, figsize=(22, 9), facecolor=BG)
fig.suptitle('LOCATION POWER MAP — Where Should You Open?',
             fontsize=17, fontweight='bold', color=B, y=0.98)

loc_stats = (df_all.groupby('location').agg(
    count=('restaurant_name','count'),
    avg_cost=('cost','mean'),
    delivery_rate=('online_delivery','mean'),
    booking_rate=('book_table','mean'),
    unique_cuisines=('cuisines','nunique')
).reset_index())
loc_stats = loc_stats[loc_stats['count'] >= 20].sort_values('count', ascending=False)

# Top 20 by count
ax = axes[0]
top20_loc = loc_stats.head(20)
ax.barh(range(len(top20_loc)), top20_loc['count'],
        color=[PAL[i%len(PAL)] for i in range(len(top20_loc))], edgecolor='none', height=0.72)
ax.set_yticks(range(len(top20_loc))); ax.set_yticklabels(top20_loc['location'], fontsize=9)
ax.invert_yaxis()
ax.set_title('TOP 20 AREAS BY COUNT', fontsize=11, color=WHITE, pad=12)
ax.set_xlabel('Number of Restaurants', fontsize=10)
ax.grid(axis='x', alpha=0.3)

# Bubble — count vs cost, color = delivery rate
ax2 = axes[1]
top30 = loc_stats.head(30)
sc = ax2.scatter(top30['count'], top30['avg_cost'],
                 s=top30['unique_cuisines']*15,
                 c=top30['delivery_rate'],
                 cmap=LinearSegmentedColormap.from_list('dr',[MUTED, B]),
                 alpha=0.85, edgecolors=BG, linewidth=0.5)
cbar = plt.colorbar(sc, ax=ax2, shrink=0.7)
cbar.set_label('Delivery Rate', color=WHITE, fontsize=9)
cbar.ax.yaxis.set_tick_params(color=MUTED)
plt.setp(plt.getp(cbar.ax.axes,'yticklabels'), color=MUTED, fontsize=8)
for _, row in top30.iterrows():
    ax2.annotate(row['location'], (row['count'], row['avg_cost']),
                 fontsize=7, color=WHITE, alpha=0.8, xytext=(4,3),
                 textcoords='offset points')
ax2.set_xlabel('Number of Restaurants', fontsize=10)
ax2.set_ylabel('Avg Cost (₹)', fontsize=10)
ax2.set_title('DENSITY vs PRICE\n(Bubble=cuisine variety)', fontsize=11, color=WHITE, pad=12)
ax2.grid(alpha=0.3)

# Delivery + booking adoption per top 15
ax3 = axes[2]
top15_loc = loc_stats.head(15)
x = range(len(top15_loc))
ax3.bar(x, top15_loc['delivery_rate']*100, color=A, edgecolor='none', label='Delivery %', alpha=0.9)
ax3.bar(x, top15_loc['booking_rate']*100,  color=B, edgecolor='none', label='Booking %',  alpha=0.9)
ax3.set_xticks(x)
ax3.set_xticklabels(top15_loc['location'], rotation=45, ha='right', fontsize=8)
ax3.set_ylabel('Adoption Rate (%)', fontsize=10)
ax3.set_title('DIGITAL ADOPTION BY AREA', fontsize=11, color=WHITE, pad=12)
ax3.legend(fontsize=9, frameon=False); ax3.grid(axis='y', alpha=0.3)

top_area = loc_stats.iloc[0]
note(fig, 0.02,
     f"Top area: {top_area['location']} — {int(top_area['count'])} restaurants, "
     f"{top_area['unique_cuisines']} cuisines, ₹{int(top_area['avg_cost'])} avg cost.", B)
plt.tight_layout(rect=[0, 0.06, 1, 0.96])
save(fig, 'viz5_location_powermap.png')


# ════════════════════════════════════════════════════════════
#  VIZ 6 — OPPORTUNITY MATRIX
# ════════════════════════════════════════════════════════════
print("[6] Opportunity Matrix...")

fig, axes = plt.subplots(1, 2, figsize=(22, 10), facecolor=BG)
fig.suptitle('OPPORTUNITY MATRIX — Which Cuisine is Underserved Where?',
             fontsize=17, fontweight='bold', color=PINK, y=0.98)

top10_cui = df_all['cuisines'].value_counts().head(10).index
top12_loc2 = df_all['location'].value_counts().head(12).index

# Count heatmap
ax = axes[0]
pivot_count = (df_all[df_all['location'].isin(top12_loc2) & df_all['cuisines'].isin(top10_cui)]
               .groupby(['location','cuisines']).size().unstack(fill_value=0))
sns.heatmap(pivot_count, ax=ax, annot=True, fmt='d',
            cmap=LinearSegmentedColormap.from_list('opp',[CARD, PINK]),
            linewidths=1, linecolor=BG, annot_kws={'size':9,'color':WHITE},
            cbar_kws={'shrink':0.8})
ax.set_title('RESTAURANT COUNT\n(zeros = opportunity gaps)', fontsize=11, color=WHITE, pad=12)
ax.tick_params(axis='x', rotation=40, labelsize=9)
ax.tick_params(axis='y', rotation=0, labelsize=9)

# Percentage heatmap
ax2 = axes[1]
pivot_pct = pivot_count.div(pivot_count.sum(axis=1), axis=0) * 100
sns.heatmap(pivot_pct, ax=ax2, annot=True, fmt='.1f',
            cmap=LinearSegmentedColormap.from_list('opp2',[CARD, GOLD, A]),
            linewidths=1, linecolor=BG, annot_kws={'size':9,'color':BG},
            cbar_kws={'shrink':0.8, 'label':'% of area restaurants'})
ax2.set_title('CUISINE SHARE PER AREA (%)\nHigh=saturated  Low=opportunity', fontsize=11, color=WHITE, pad=12)
ax2.tick_params(axis='x', rotation=40, labelsize=9)
ax2.tick_params(axis='y', rotation=0, labelsize=9)

note(fig, 0.02,
     "Cells near zero = untapped. Find your cuisine+location combo where competition is low but footfall is high.", PINK)
plt.tight_layout(rect=[0, 0.05, 1, 0.96])
save(fig, 'viz6_opportunity_matrix.png')


# ════════════════════════════════════════════════════════════
#  VIZ 7 — FEATURE ADOPTION
# ════════════════════════════════════════════════════════════
print("[7] Feature Adoption...")

fig, axes = plt.subplots(2, 2, figsize=(18, 13), facecolor=BG)
fig.suptitle('FEATURE ADOPTION — Delivery & Booking Intelligence',
             fontsize=17, fontweight='bold', color=PURPLE, y=0.98)

# Delivery by cuisine
ax = axes[0][0]
top12 = df_all['cuisines'].value_counts().head(12).index
del_rate = (df_all[df_all['cuisines'].isin(top12)]
            .groupby('cuisines')['online_delivery'].mean().sort_values(ascending=True))
ax.barh(range(len(del_rate)), del_rate.values*100,
        color=[PAL[i%len(PAL)] for i in range(len(del_rate))], edgecolor='none', height=0.65)
ax.set_yticks(range(len(del_rate))); ax.set_yticklabels(del_rate.index, fontsize=9)
ax.axvline(50, color=WHITE, linestyle='--', alpha=0.4, label='50% mark')
ax.set_xlabel('Delivery Adoption (%)', fontsize=10)
ax.set_title('DELIVERY ADOPTION BY CUISINE', fontsize=11, color=WHITE, pad=12)
ax.legend(fontsize=8, frameon=False); ax.grid(axis='x', alpha=0.3)

# Booking by location
ax2 = axes[0][1]
top12_loc = df_all['location'].value_counts().head(12).index
book_rate = (df_all[df_all['location'].isin(top12_loc)]
             .groupby('location')['book_table'].mean().sort_values(ascending=True))
ax2.barh(range(len(book_rate)), book_rate.values*100,
         color=[PAL[i%len(PAL)] for i in range(len(book_rate))], edgecolor='none', height=0.65)
ax2.set_yticks(range(len(book_rate))); ax2.set_yticklabels(book_rate.index, fontsize=9)
ax2.axvline(df_all['book_table'].mean()*100, color=WHITE, linestyle='--',
            alpha=0.4, label=f'Avg {df_all["book_table"].mean()*100:.1f}%')
ax2.set_xlabel('Booking Adoption (%)', fontsize=10)
ax2.set_title('BOOKING ADOPTION BY LOCATION', fontsize=11, color=WHITE, pad=12)
ax2.legend(fontsize=8, frameon=False); ax2.grid(axis='x', alpha=0.3)

# Feature combo count matrix
ax3 = axes[1][0]
combo = df_all.groupby(['online_delivery','book_table']).size().unstack(fill_value=0)
combo.index = ['No Delivery','Has Delivery']
combo.columns = ['No Booking','Has Booking']
sns.heatmap(combo, ax=ax3, annot=True, fmt=',d',
            cmap=LinearSegmentedColormap.from_list('pm',[CARD, PURPLE]),
            linewidths=3, linecolor=BG,
            annot_kws={'size':16,'color':WHITE,'fontweight':'bold'},
            cbar_kws={'shrink':0.8})
ax3.set_title('FEATURE COMBINATION MATRIX\n(# Restaurants)', fontsize=11, color=WHITE, pad=12)
ax3.tick_params(labelsize=10)

# Avg cost by feature combo
ax4 = axes[1][1]
combos = [(0,0,'Neither',MUTED),(1,0,'Delivery Only',A),(0,1,'Booking Only',B),(1,1,'Both',GOLD)]
lc, mc, cc, rc = [], [], [], []
for d, b, lbl, clr in combos:
    sub = df_all[(df_all['online_delivery']==d)&(df_all['book_table']==b)]['cost']
    if len(sub) > 0:
        lc.append(lbl); mc.append(sub.mean()); cc.append(len(sub)); rc.append(clr)
bars4 = ax4.bar(lc, mc, color=rc, edgecolor='none', width=0.55)
for bar, v, cnt in zip(bars4, mc, cc):
    ax4.text(bar.get_x()+bar.get_width()/2, v+8,
             f'₹{int(v)}\nn={cnt:,}', ha='center', fontsize=10,
             color=WHITE, fontweight='bold', linespacing=1.5)
ax4.set_ylabel('Avg Cost (₹)', fontsize=10)
ax4.set_title('AVG PRICE BY FEATURE COMBO', fontsize=11, color=WHITE, pad=12)
ax4.grid(axis='y', alpha=0.3)

note(fig, 0.01,
     "Restaurants with BOTH features charge ~40% more — features signal premium positioning.", PURPLE)
plt.tight_layout(rect=[0, 0.04, 1, 0.96])
save(fig, 'viz7_feature_adoption.png')


# ════════════════════════════════════════════════════════════
#  VIZ 8 — RESTAURANT CLUSTERING
# ════════════════════════════════════════════════════════════
print("[8] Restaurant Clustering...")

df8 = df_all.copy()
df8['cuisine_enc']  = LabelEncoder().fit_transform(df8['cuisines'].fillna('Unknown'))
df8['location_enc'] = LabelEncoder().fit_transform(df8['location'].fillna('Unknown'))
df8['cost_norm']    = (df8['cost'] - df8['cost'].min()) / (df8['cost'].max() - df8['cost'].min())

X8 = df8[['cost_norm','online_delivery','book_table','cuisine_enc']].fillna(0)
km = KMeans(n_clusters=4, random_state=42, n_init=10)
df8['cluster'] = km.fit_predict(X8)
cluster_colors = [A, B, GOLD, PINK]

fig, axes = plt.subplots(1, 3, figsize=(22, 8), facecolor=BG)
fig.suptitle('RESTAURANT DNA — 4 Archetypes Across Bangalore',
             fontsize=17, fontweight='bold', color=GOLD, y=0.98)

# Scatter
ax = axes[0]
labels_k = ['Budget Street Food', 'Mid-Range Diner', 'Premium Dining', 'Delivery-First']
for ci in range(4):
    sub = df8[df8['cluster']==ci]
    ax.scatter(sub['cost'], sub['online_delivery'] + np.random.normal(0, 0.05, len(sub)),
               alpha=0.3, s=12, color=cluster_colors[ci],
               label=labels_k[ci], edgecolors='none')
ax.set_xlabel('Cost for Two (₹)', fontsize=10)
ax.set_ylabel('Online Delivery (0/1)', fontsize=10)
ax.set_title('CLUSTER SCATTER', fontsize=11, color=WHITE, pad=12)
ax.legend(fontsize=8, frameon=False, markerscale=2)
ax.set_xlim(0, 2500); ax.grid(alpha=0.3)

# Profile bars
ax2 = axes[1]
metrics = ['Avg Cost\n(÷100)', 'Delivery\nRate %', 'Booking\nRate %', 'Size\n(÷100)']
x = np.arange(len(metrics)); w = 0.18
for i, ci in enumerate(range(4)):
    sub = df8[df8['cluster']==ci]
    vals = [sub['cost'].mean()/100, sub['online_delivery'].mean()*100,
            sub['book_table'].mean()*100, len(sub)/100]
    ax2.bar(x + i*w - 1.5*w, vals, w, color=cluster_colors[ci],
            edgecolor='none', label=labels_k[ci], alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels(metrics, fontsize=9)
ax2.set_title('CLUSTER PROFILES', fontsize=11, color=WHITE, pad=12)
ax2.legend(fontsize=8, frameon=False, loc='upper right')
ax2.grid(axis='y', alpha=0.3)

# Sizes + top cuisine
ax3 = axes[2]
for ci in range(4):
    sub = df8[df8['cluster']==ci]
    top_c = sub['cuisines'].value_counts().index[0]
    ax3.barh(ci, len(sub), color=cluster_colors[ci], edgecolor='none', height=0.55)
    ax3.text(len(sub)+20, ci,
             f"{labels_k[ci]}  |  {top_c}  |  n={len(sub):,}",
             va='center', fontsize=9, color=WHITE)
ax3.set_yticks(range(4)); ax3.set_yticklabels([f'Cluster {i}' for i in range(4)], fontsize=9)
ax3.set_xlabel('Number of Restaurants', fontsize=10)
ax3.set_title('CLUSTER SIZES + TOP CUISINE', fontsize=11, color=WHITE, pad=12)
ax3.set_xlim(0, df8['cluster'].value_counts().max()*1.7)
ax3.grid(axis='x', alpha=0.3); ax3.invert_yaxis()

note(fig, 0.02, "4 restaurant archetypes emerge from the data. Know which cluster you're competing in.", GOLD)
plt.tight_layout(rect=[0, 0.06, 1, 0.96])
save(fig, 'viz8_restaurant_clusters.png')


# ════════════════════════════════════════════════════════════
#  VIZ 9 — ML SUCCESS PREDICTOR
# ════════════════════════════════════════════════════════════
print("[9] ML Success Predictor...")

df9 = df_all.copy()
df9['cuisine_enc']  = LabelEncoder().fit_transform(df9['cuisines'].fillna('Unknown'))
df9['location_enc'] = LabelEncoder().fit_transform(df9['location'].fillna('Unknown'))
df9['is_premium']   = (
    (df9['cost'] >= df9['cost'].quantile(0.6)).astype(int) +
    df9['online_delivery'] + df9['book_table'] >= 2
).astype(int)

feature_cols = ['cuisine_enc','location_enc','cost','online_delivery','book_table']
X9 = df9[feature_cols].fillna(0)
y9 = df9['is_premium']
X_train, X_test, y_train, y_test = train_test_split(X9, y9, test_size=0.2, random_state=42)
rf9 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf9.fit(X_train, y_train)
acc = rf9.score(X_test, y_test)

importances = pd.Series(rf9.feature_importances_, index=feature_cols).sort_values()
name_map = {'cuisine_enc':'Cuisine Type','location_enc':'Location',
            'cost':'Price Point','online_delivery':'Online Delivery','book_table':'Table Booking'}
importances.index = [name_map[i] for i in importances.index]

fig, axes = plt.subplots(1, 2, figsize=(18, 7), facecolor=BG)
fig.suptitle(f'ML MODEL — What Predicts a Premium Restaurant? (Accuracy: {acc*100:.1f}%)',
             fontsize=17, fontweight='bold', color=A, y=0.98)

ax = axes[0]
bars = ax.barh(importances.index, importances.values,
               color=[PAL[i] for i in range(len(importances))], edgecolor='none', height=0.55)
ax.set_xlabel('Feature Importance Score', fontsize=10)
ax.set_title('WHAT DRIVES PREMIUM POSITIONING?', fontsize=11, color=WHITE, pad=12)
ax.grid(axis='x', alpha=0.3)
for bar, val in zip(bars, importances.values):
    ax.text(val+0.002, bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10, color=WHITE)

ax2 = axes[1]
price_bins = pd.cut(df9['cost'], bins=10)
premium_by_price = df9.groupby(price_bins, observed=True)['is_premium'].mean()*100
bin_centers = [(b.left+b.right)/2 for b in premium_by_price.index]
ax2.plot(bin_centers, premium_by_price.values, color=GOLD, linewidth=3, marker='o', markersize=8)
ax2.fill_between(bin_centers, premium_by_price.values, alpha=0.2, color=GOLD)
ax2.axhline(50, color=MUTED, linestyle='--', alpha=0.5, label='50% threshold')
ax2.set_xlabel('Cost for Two (₹)', fontsize=10)
ax2.set_ylabel('% Classified as Premium', fontsize=10)
ax2.set_title('HOW PRICE PREDICTS PREMIUM STATUS', fontsize=11, color=WHITE, pad=12)
ax2.legend(fontsize=9, frameon=False); ax2.grid(alpha=0.3)

note(fig, 0.02,
     f"Top factor: {importances.idxmax()}. That's the #1 lever for your friend's restaurant.", A)
plt.tight_layout(rect=[0, 0.06, 1, 0.96])
save(fig, 'viz9_ml_predictor.png')


# ════════════════════════════════════════════════════════════
#  VIZ 10 — SMART RECOMMENDER
# ════════════════════════════════════════════════════════════
print("[10] Smart Recommender...")

def smart_recommend(cuisine_pref, budget, location_pref=None, top_n=6):
    d = df_all.copy()
    results = d[d['cuisines'].str.lower() == cuisine_pref.lower()]
    results = results[results['cost'].between(budget*0.5, budget*1.4)]
    if location_pref:
        loc_match = results[results['location'].str.lower() == location_pref.lower()]
        if len(loc_match) >= 3:
            results = loc_match
    if len(results) == 0:
        results = d[d['cost'].between(budget*0.5, budget*1.5)]
    results = results.copy()
    results['cost_score']    = 1 - abs(results['cost'] - budget) / max(budget, 1)
    results['feature_score'] = (results['online_delivery'] + results['book_table']) / 2
    results['final_score']   = results['cost_score']*0.5 + results['feature_score']*0.5
    return (results.sort_values('final_score', ascending=False)
                   .drop_duplicates('restaurant_name').head(top_n))

test_profiles = [
    {'label': 'Student',      'cuisine': 'Fast Food',    'budget': 250,  'loc': 'Btm'},
    {'label': 'Professional', 'cuisine': 'Continental',  'budget': 900,  'loc': 'Indiranagar'},
    {'label': 'Family',       'cuisine': 'South Indian', 'budget': 500,  'loc': 'Whitefield'},
    {'label': 'Date Night',   'cuisine': 'North Indian', 'budget': 1200, 'loc': 'Koramangala'},
]

fig = plt.figure(figsize=(22, 14), facecolor=BG)
fig.suptitle('SMART RESTAURANT RECOMMENDER — 4 User Profiles',
             fontsize=17, fontweight='bold', color=B, y=0.98)
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3)

for idx, profile in enumerate(test_profiles):
    row, col = idx//2, idx%2
    ax = fig.add_subplot(gs[row, col])
    ax.set_facecolor(CARD2)
    recs = smart_recommend(profile['cuisine'], profile['budget'], profile['loc'], top_n=6)
    color = PAL[idx]
    ax.set_xlim(0, 10); ax.set_ylim(-0.5, len(recs)+0.8); ax.axis('off')
    ax.text(5, len(recs)+0.5,
            f"{profile['label']}  ·  {profile['cuisine']}  ·  <= Rs{profile['budget']}  ·  {profile['loc']}",
            ha='center', fontsize=11, color=color, fontweight='bold')
    for i, (_, row_r) in enumerate(recs.iterrows()):
        y     = len(recs) - i - 0.5
        score = row_r.get('final_score', 0)
        cost  = row_r.get('cost', 0)
        name  = str(row_r['restaurant_name'])[:26]
        loc   = str(row_r['location'])[:18]
        has_d = '[D]' if row_r['online_delivery'] else '   '
        has_b = '[B]' if row_r['book_table'] else '   '
        ax.barh(y, 10, height=0.65, color='#111111', left=0)
        ax.barh(y, score*10, height=0.65, color=color, alpha=0.35, left=0)
        ax.text(0.4, y, f'#{i+1}', va='center', ha='center', fontsize=12,
                color=color, fontweight='bold')
        ax.text(0.9, y+0.18, name, va='center', fontsize=9.5, color=WHITE, fontweight='bold')
        ax.text(0.9, y-0.18, f'{loc}  ·  Rs{int(cost)}  {has_d}{has_b}',
                va='center', fontsize=8, color=MUTED)
        ax.text(9.5, y, f'{score:.2f}', va='center', ha='center',
                fontsize=9, color=GOLD, fontweight='bold')

note(fig, 0.01,
     "Score = 50% cost proximity + 50% feature score. [D]=delivery available, [B]=booking available.", B)
save(fig, 'viz10_smart_recommender.png')


# ════════════════════════════════════════════════════════════
#  VIZ 0 — MASTER DASHBOARD
# ════════════════════════════════════════════════════════════
print("[0] Master Dashboard...")

fig = plt.figure(figsize=(22, 12), facecolor=BG)
fig.text(0.5, 0.93, 'ZOMATO UNCOVERED', ha='center',
         fontsize=42, fontweight='bold', color=A, fontfamily='monospace')
fig.text(0.5, 0.87, 'The Data Behind Every Dish  —  Bangalore Restaurant Intelligence',
         ha='center', fontsize=14, color=MUTED, style='italic')
fig.add_artist(plt.Line2D([0.05,0.95],[0.84,0.84], color=A, linewidth=1.5,
                           transform=fig.transFigure))

stats = [
    ('RESTAURANTS', f"{len(df_all):,}",                              A),
    ('LOCATIONS',   f"{df_all['location'].nunique()}",               B),
    ('CUISINES',    f"{df_all['cuisines'].nunique()}",               GOLD),
    ('AVG COST',    f"Rs{int(df_all['cost'].mean())}",               PINK),
    ('RATED',       f"{df_all['rating'].notna().sum():,}",           PURPLE),
    ('AVG RATING',  f"{df_all['rating'].mean():.2f}" if has_rating else 'N/A', BLUE),
]
for i, (label, value, color) in enumerate(stats):
    x = 0.09 + i*0.145
    fig.text(x, 0.75, value, ha='center', fontsize=28, fontweight='bold', color=color)
    fig.text(x, 0.67, label, ha='center', fontsize=9, color=MUTED)

fig.add_artist(plt.Line2D([0.05,0.95],[0.63,0.63], color=GRID, linewidth=1,
                           transform=fig.transFigure))

vizzes = [
    ('VIZ 1', 'Dataset\nOverview',     A),
    ('VIZ 2', 'Cuisine\nDeep Dive',    B),
    ('VIZ 3', 'Price\nIntelligence',   GOLD),
    ('VIZ 4', 'Ratings\nAnalysis',     PINK),
    ('VIZ 5', 'Location\nPower Map',   PURPLE),
    ('VIZ 6', 'Opportunity\nMatrix',   BLUE),
    ('VIZ 7', 'Feature\nAdoption',     '#34d399'),
    ('VIZ 8', 'Restaurant\nClusters',  '#fb923c'),
    ('VIZ 9', 'ML Predictor',          '#e879f9'),
    ('VIZ 10','Recommender',           A),
]
for i, (num, title, color) in enumerate(vizzes):
    x_center = 0.07 + i*0.093
    ax_card = fig.add_axes([x_center-0.038, 0.16, 0.076, 0.42])
    ax_card.set_facecolor(color+'18')
    for spine in ax_card.spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(1.5)
    ax_card.set_xticks([]); ax_card.set_yticks([])
    fig.text(x_center, 0.54, num, ha='center', fontsize=9,
             fontweight='bold', color=color, fontfamily='monospace')
    fig.text(x_center, 0.36, title, ha='center', fontsize=8.5,
             color=WHITE, linespacing=1.6, fontfamily='monospace')

fig.text(0.5, 0.09, 'Sources: restaurant_cleaned_FINAL.csv  +  restaurant_final_v3_perfect.csv',
         ha='center', fontsize=9, color=MUTED)
fig.text(0.5, 0.05,
         'Covers: Overview · Cuisine · Price · Ratings · Location · Opportunity · '
         'Features · Clustering · ML · Recommender',
         ha='center', fontsize=9, color=MUTED)

save(fig, 'viz0_master_dashboard.png')

print("\n" + "="*55)
print("  ALL 11 CHARTS GENERATED")
print("="*55)
outputs = [
    ('viz0_master_dashboard.png',    'Overview index'),
    ('viz1_dataset_overview.png',    'Data summary'),
    ('viz2_cuisine_deepdive.png',    'Cuisine supply + cost + share'),
    ('viz3_price_intelligence.png',  'Price tiers, boxplots, heatmap'),
    ('viz4_ratings_analysis.png',    'What gets rated highly'),
    ('viz5_location_powermap.png',   'Where to open'),
    ('viz6_opportunity_matrix.png',  'Underserved gaps'),
    ('viz7_feature_adoption.png',    'Delivery & booking intel'),
    ('viz8_restaurant_clusters.png', '4 restaurant archetypes'),
    ('viz9_ml_predictor.png',        'ML feature importance'),
    ('viz10_smart_recommender.png',  '4 user recommendations'),
]
for fname, desc in outputs:
    print(f"  {fname:<35} {desc}")

Info:    12,853 rows
Ratings: 2,171 rows
Merged:  12,120 rows | with rating: 2,162

[1] Dataset Overview...
   ✓ viz1_dataset_overview.png
[2] Cuisine Deep Dive...
   ✓ viz2_cuisine_deepdive.png
[3] Price Intelligence...
   ✓ viz3_price_intelligence.png
[4] Ratings Analysis...
   ✓ viz4_ratings_analysis.png
[5] Location Power Map...
   ✓ viz5_location_powermap.png
[6] Opportunity Matrix...
   ✓ viz6_opportunity_matrix.png
[7] Feature Adoption...
   ✓ viz7_feature_adoption.png
[8] Restaurant Clustering...
   ✓ viz8_restaurant_clusters.png
[9] ML Success Predictor...
   ✓ viz9_ml_predictor.png
[10] Smart Recommender...
   ✓ viz10_smart_recommender.png
[0] Master Dashboard...
   ✓ viz0_master_dashboard.png

  ALL 11 CHARTS GENERATED
  viz0_master_dashboard.png           Overview index
  viz1_dataset_overview.png           Data summary
  viz2_cuisine_deepdive.png           Cuisine supply + cost + share
  viz3_price_intelligence.png         Price tiers, boxplots, heatmap
  viz4_ratings_anal